# Regime detector — one-click check

Set the knobs in the next cell, then **Run All**. Everything below is causal:
the detector only ever sees bars `<= t`. The label table and the entry scan look
into the future *on purpose* and never enter the filter.

Every threshold is in **driftless-random-walk units** — `slope_z` is standard
normal when nothing is happening, and `er_rw = ER * sqrt(n)` has mean 1.0 for
any window length. So `entry_z = 2` means the same thing on TSLA as on JPM.


In [1]:
# ---- knobs -----------------------------------------------------------------
SYMBOL   = "AAPL"          # QQQ drops ~4 minutes a session on IEX; its labels are sparse
START    = "2026-06-01"
END      = "2026-08-01"
PRESET   = "balanced"      # sensitive | balanced | conservative
OVERRIDES: dict = {}       # e.g. {"entry_z": 1.5, "er_entry_rw": 1.5}
PLOT_DAY = None            # "2026-07-15" to chart one session, None for the tail
# -----------------------------------------------------------------------------

from pathlib import Path
import os, sys

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "qtrader").is_dir() and (_p / "config").is_dir():
        REPO_ROOT = _p
        break
else:
    raise ModuleNotFoundError("Cannot find the qtrader repo")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

from qtrader.data.ingest import load_clean_bars
from qtrader.regime import (
    KalmanCUSUMConfig,
    KalmanCUSUMRegimeDetector,
    describe_entries,
    evaluate_regime,
    null_entry_rate,
)

CFG = KalmanCUSUMConfig.from_preset(PRESET).replace(**OVERRIDES)
BARS = load_clean_bars(SYMBOL, timeframe="1Min", feed="iex", start=START, end=END)
RESULT = KalmanCUSUMRegimeDetector(CFG).run(BARS)
print(f"{SYMBOL}  {len(BARS):,} bars  {BARS.index[0].date()} -> {BARS.index[-1].date()}")
print(RESULT["state"].value_counts(normalize=True).round(3).to_dict())


AAPL  16,770 bars  2026-06-01 -> 2026-07-31
{'FLAT': 0.918, 'DOWN': 0.045, 'UP': 0.037}


## 1. Is the scale honest?

`slope_z` should have sd ≈ 1 and `P(|z| > 2)` ≈ 0.046 — that is the whole point
of standardising by the Lyapunov null scale rather than by the Kalman posterior
`sqrt(P[1,1])`, whose spread ran from 1.4 (JPM) to 2.0 (TSLA).

If the sd here is far from 1 on your symbol, the thresholds do not mean what
they say and nothing below is interpretable.


In [2]:
z = RESULT["slope_z"].dropna()
er = RESULT["er_rw"].dropna()
print(f"slope_z      sd = {z.std():.3f}   (target 1.00)")
print(f"             P(|z| > 2) = {(z.abs() > 2).mean():.3f}   (target 0.046)")
print(f"             lag-1 autocorr = {z.autocorr(1):.3f}   (a filtered series is meant to be smooth)")
print(f"er_rw        median = {er.median():.2f} over {RESULT['er_bars'].dropna().median():.0f}-bar windows")
print(f"             (null median 0.88, null q90 2.05, for ANY window length)")


slope_z      sd = 1.060   (target 1.00)
             P(|z| > 2) = 0.060   (target 0.046)
             lag-1 autocorr = 0.943   (a filtered series is meant to be smooth)
er_rw        median = 0.85 over 10-bar windows
             (null median 0.88, null q90 2.05, for ANY window length)


## 2. What does this threshold set cost when nothing is happening?

Replayed over simulated driftless random walks, so **every** entry is false by
construction. This number belongs to the thresholds alone — no symbol, no feed,
no label configuration.


In [3]:
rows = []
for name in ("sensitive", "balanced", "conservative"):
    m = null_entry_rate(KalmanCUSUMConfig.from_preset(name), n_sessions=60)
    rows.append({"preset": name, **{k: round(v, 2) for k, v in m.items() if k != "n_sessions"}})
rows.append({"preset": f"yours ({PRESET}+overrides)",
             **{k: round(v, 2) for k, v in null_entry_rate(CFG, n_sessions=60).items() if k != "n_sessions"}})
pd.DataFrame(rows)


,preset,null_entries_per_session,null_frac_in_trend,null_flips_per_session,null_mean_trend_duration
0,sensitive,7.78,0.22,15.20,11.24
1,balanced,2.98,0.09,5.88,11.34
2,conservative,0.83,0.02,1.67,11.32
3,yours (balanced+overrides),2.98,0.09,5.88,11.34


## 3. Does the state describe the move it claims to?

**This is the test that matters for "感知当前走势状态".**

* `offset < 0` — the move the detector was reacting to. A correct state label
  makes this large and positive; a random walk would give 0.
* `offset > 0` — what happened *after* the state was declared. This is a
  forecast claim, and a regime detector does not make one. Near-zero here is the
  expected, correct result: the state belongs in a gate or a veto, never in an
  alpha score.


In [4]:
scan = describe_entries(BARS, RESULT)
print(f"{scan.attrs['n_entries']} UP/DOWN entries")
scan.round(3)


138 UP/DOWN entries


,offset,kind,n,mean_move_rw,frac_right_way
0,-30,before (what it is describing),129,0.931,0.930
1,-20,before (what it is describing),131,1.091,0.992
2,-10,before (what it is describing),138,1.782,1.000
3,-5,before (what it is describing),138,1.727,1.000
4,5,after (a forecast claim),133,-0.007,0.481
5,10,after (a forecast claim),127,-0.039,0.496
6,20,after (a forecast claim),122,-0.035,0.459
7,30,after (a forecast claim),118,0.015,0.517


## 4. Delay and false alarms against the offline labels

`detect_trend_events` looks into the future on purpose and never enters the
filter. It needs gap-free minutes inside `vol_lookback` *and* `horizon`, so on a
feed with missing bars it can score nothing at all — that raises now instead of
returning a quiet frame of NaNs.


In [5]:
from qtrader.labels.trend_events import detect_trend_events

_, EVENTS = detect_trend_events(BARS, SYMBOL)
if EVENTS.empty:
    print(f"no labelled events for {SYMBOL} on this window — the minutes are too gappy.")
    print("Path statistics only:")
    metrics = evaluate_regime(RESULT, None)
else:
    metrics = evaluate_regime(RESULT, EVENTS)
pd.Series({k: round(v, 3) for k, v in metrics.items()}).to_frame("value")


,value
n_bars,16770.000
n_sessions,43.000
n_flips,264.000
flips_per_session,6.140
flips_per_hour,0.945
n_up_entries,63.000
n_down_entries,75.000
mean_up_duration,9.905
mean_down_duration,10.093
mean_trend_duration,10.007


## 5. The chart

Reference lines come from `CFG`, so the chart cannot draw a threshold the
detector never applied.


In [6]:
from qtrader.viz import plot_regime

if PLOT_DAY is None:
    b, r = BARS.tail(390), RESULT.tail(390)
else:
    day = pd.Timestamp(PLOT_DAY).tz_localize("UTC")
    mask = (BARS.index >= day) & (BARS.index < day + pd.Timedelta(days=1))
    b, r = BARS[mask], RESULT[mask]

plot_regime(b, r, config=CFG, title=f"{SYMBOL} — {PRESET} {OVERRIDES or ''}")


## 6. Sweep a knob

`entry_z` and `er_entry_rw` dominate; `cusum_k` / `cusum_h` are nearly inert now
that `slope_z` is properly scaled (varying `cusum_h` from 1.5 to 4.0 moved
capture by 0.00 and the null entry rate by 0.07). Change `GRID` to explore.


In [7]:
GRID = [{"entry_z": ez, "er_entry_rw": er}
        for ez in (1.0, 1.5, 2.0, 2.5)
        for er in (1.0, 1.5, 2.0, 2.5)]

rows = []
for cell in GRID:
    cfg = CFG.replace(**cell)
    res = KalmanCUSUMRegimeDetector(cfg).run(BARS)
    m = evaluate_regime(res, None if EVENTS.empty else EVENTS)
    before = describe_entries(BARS, res)
    lead = before.loc[before.offset == -10, "mean_move_rw"]
    rows.append({
        **cell,
        "flips/sess": round(m["flips_per_session"], 1),
        "mean_dur": round(m["mean_trend_duration"], 1),
        "frac_flat": round(m["frac_flat"], 2),
        "capture": round(m["capture_ratio"], 2),
        "delay": m["median_delay"],
        "lead_10b_rw": round(float(lead.iloc[0]), 2) if len(lead) else np.nan,
        "null_entries/sess": round(null_entry_rate(cfg, n_sessions=40)["null_entries_per_session"], 2),
    })
pd.DataFrame(rows).sort_values("null_entries/sess")


,entry_z,er_entry_rw,flips/sess,mean_dur,frac_flat,capture,delay,lead_10b_rw,null_entries/sess
15,2.5,2.5,2.3,10.0,0.97,0.16,24.0,2.61,0.88
14,2.5,2.0,3.2,9.4,0.96,0.33,24.0,2.14,1.07
12,2.5,1.0,3.7,9.5,0.95,0.34,24.0,2.08,1.15
13,2.5,1.5,3.5,9.5,0.95,0.34,24.0,2.10,1.15
11,2.0,2.5,3.8,10.0,0.95,0.33,22.0,2.21,2.02
10,2.0,2.0,6.1,10.0,0.92,0.53,23.0,1.78,3.20
7,1.5,2.5,6.0,10.1,0.92,0.47,19.0,2.07,3.30
9,2.0,1.5,7.7,9.8,0.90,0.64,21.0,1.66,3.65
8,2.0,1.0,8.2,9.9,0.89,0.66,21.5,1.60,3.73
3,1.0,2.5,7.2,10.3,0.90,0.57,19.0,1.92,3.95
